In [2]:
import pandas as pd

train = pd.read_csv("../data/train_test.csv")
val = pd.read_csv("../data/validation.csv")

print(train.shape, val.shape)
train.head()

(48000, 14) (12000, 13)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [3]:
print(train.dtypes)
print()
print("Missing values per column:")
print(train.isna().sum())
print()
print("Duplicate rows:", train.duplicated().sum())
print("Duplicate load_ids:", train["load_id"].duplicated().sum())

load_id             str
pickup              str
delivery            str
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment           str
weight          float64
date                str
market_index    float64
quote_signal    float64
posted_rate     float64
dtype: object

Missing values per column:
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64

Duplicate rows: 0
Duplicate load_ids: 0


In [4]:
train.describe().T

,count,mean,std,min,25%,50%,75%,max
pickup_lat,48000.0,35.647545,4.315285,28.35765,31.98691,35.29479,39.411040,44.30296
pickup_lon,48000.0,-90.928964,13.482431,-121.69849,-98.40059,-88.08915,-83.285060,-69.50000
delivery_lat,48000.0,35.641175,4.317199,28.35765,31.98691,35.29479,39.411040,44.30296
delivery_lon,48000.0,-90.857310,13.476589,-121.69849,-98.40059,-87.52871,-83.285060,-69.50000
distance,48000.0,1135.856654,728.564416,70.00000,550.40000,953.30000,1645.525000,3439.80000
weight,47700.0,31028.844004,9391.440620,-47500.00000,25800.00000,31436.50000,37018.000000,47500.00000
market_index,47626.0,1.083387,0.168091,0.67639,0.94967,1.05580,1.219590,1.46778
quote_signal,48000.0,2.062468,0.291391,0.69228,1.89103,2.05575,2.221685,3.61035
posted_rate,48000.0,2373.980682,1486.493245,57.22000,1251.55500,2030.76000,3330.750000,25533.00000


In [5]:
print("Negative weights:", (train["weight"] < 0).sum())
print(train.loc[train["weight"] < 0, ["distance", "equipment", "weight", "posted_rate"]].head())
print()
print("Top 10 rates:")
print(train.nlargest(10, "posted_rate")[["pickup", "delivery", "distance", "equipment", "posted_rate"]])
print()
print("Rate per mile summary:")
print((train["posted_rate"] / train["distance"]).describe())

Negative weights: 292
     distance equipment   weight  posted_rate
68     1334.2    Reefer -36559.0      2831.27
204     496.2   Dry Van -26670.0      1046.92
206     411.3    Reefer -20003.0       959.15
225    1317.4    Reefer -23981.0      2836.00
376    1331.1   Dry Van -41397.0      2522.54

Top 10 rates:
            pickup       delivery  distance equipment  posted_rate
12184  Bakersfield       Hartford    2829.8   Dry Van     25533.00
37764      Phoenix       Syracuse    2810.9   Dry Van     24294.98
1466   Los Angeles       Syracuse    2786.0    Reefer     24140.21
17371         Reno   Philadelphia    2777.6    Reefer     23662.71
26235   Charleston        Phoenix    2552.5   Dry Van     23580.42
28219       Tucson        Raleigh    2423.3   Dry Van     22755.66
3353       Phoenix     Charleston    2527.9    Reefer     22534.65
40171   Montgomery         Fresno    2283.4   Dry Van     20361.89
40096    Baltimore  Oklahoma City    1760.3    Reefer     20132.37
47298       Bosto

In [6]:
train["rpm"] = train["posted_rate"] / train["distance"]

print("Rate per mile quantiles:")
print(train["rpm"].quantile([0.5, 0.9, 0.99, 0.999]))
print()
print("Rows above $4/mile:", (train["rpm"] > 4).sum())
print("Rows above $6/mile:", (train["rpm"] > 6).sum())
print()
print(train[train["rpm"] > 4][["date", "equipment", "distance", "market_index", "quote_signal", "rpm"]].head(10))
print()
print("Median rate per mile, negative vs normal weight:")
print(train.groupby(train["weight"] < 0)["rpm"].median())

Rate per mile quantiles:
0.500     2.145348
0.900     2.568876
0.990     3.176940
0.999    10.068775
Name: rpm, dtype: float64

Rows above $4/mile: 340
Rows above $6/mile: 270

            date equipment  distance  market_index  quote_signal        rpm
388   2025-01-03   Flatbed     823.0       0.92409       2.22467   6.665237
469   2025-01-04   Dry Van    2154.3       0.84517       1.76215   5.038616
785   2025-01-06   Dry Van     861.5       0.83097       1.91323   5.325398
870   2025-01-06    Reefer     305.6       0.80857       2.64181  13.315576
1149  2025-01-08   Flatbed     170.3       0.97019       2.77328   7.983558
1175  2025-01-08   Dry Van     260.5       0.96147       2.24710   6.177428
1329  2025-01-09   Dry Van    1190.5       0.98010       1.87616   5.964494
1415  2025-01-09    Reefer     198.2       0.99663       2.66449  12.663774
1417  2025-01-09   Flatbed     613.6       1.00981       2.25857   7.310544
1466  2025-01-10    Reefer    2786.0       0.98678       1.8537

In [7]:
train["month"] = pd.to_datetime(train["date"]).dt.month
train["distance_bucket"] = pd.cut(train["distance"], [0, 250, 500, 1000, 2000, 4000])
train["rpm_ratio"] = train["rpm"] / train.groupby("distance_bucket", observed=True)["rpm"].transform("median")

print("Rows by rpm_ratio range:")
print(pd.cut(train["rpm_ratio"], [0, 1.5, 2, 2.5, 3, 4, 6, 20]).value_counts().sort_index())
print()

flag = train["rpm_ratio"] > 2.0
print("Flagged rows:", flag.sum())
print()
print("Flagged by month:")
print(train[flag].groupby("month").size())
print()
print("Flagged by equipment:")
print(train[flag].groupby("equipment").size())

Rows by rpm_ratio range:
rpm_ratio
(0.0, 1.5]     47660
(1.5, 2.0]         0
(2.0, 2.5]        36
(2.5, 3.0]        72
(3.0, 4.0]       122
(4.0, 6.0]       110
(6.0, 20.0]        0
Name: count, dtype: int64

Flagged rows: 340

Flagged by month:
month
1     41
2     29
3     34
4     30
5     25
6     39
7     36
8     33
9     34
10    39
dtype: int64

Flagged by equipment:
equipment
Dry Van    197
Flatbed     63
Reefer      80
dtype: int64


In [8]:
print("Validation missing values:")
print(val[["weight", "market_index"]].isna().sum())
print()

train["date_dt"] = pd.to_datetime(train["date"])
print("market_index: how much does it vary within one day?")
print(train.groupby("date_dt")["market_index"].std().describe())
print()

print("market_index: how much does the daily average vary across the year?")
print(train.groupby("date_dt")["market_index"].mean().describe())
print()

print("Median rate per mile, missing vs present weight:")
print(train.groupby(train["weight"].isna())["rpm"].median())

Validation missing values:
weight          165
market_index    249
dtype: int64

market_index: how much does it vary within one day?
count    304.000000
mean       0.024939
std        0.001423
min        0.020531
25%        0.024024
50%        0.024850
75%        0.025815
max        0.029482
Name: market_index, dtype: float64

market_index: how much does the daily average vary across the year?
count    304.000000
mean       1.082026
std        0.166524
min        0.766640
25%        0.946007
50%        1.051789
75%        1.212359
max        1.401592
Name: market_index, dtype: float64

Median rate per mile, missing vs present weight:
weight
False    2.145602
True     2.117580
Name: rpm, dtype: float64


In [10]:
print("Negative weights in validation:", (val["weight"] < 0).sum())
print()
print(val[["distance", "weight", "market_index", "quote_signal"]].describe().T)
print()

train_cities = set(train["pickup"]) | set(train["delivery"])
val_cities = set(val["pickup"]) | set(val["delivery"])
print("Cities in validation but not in train:", val_cities - train_cities)
print("Equipment types train:", sorted(train["equipment"].unique()))
print("Equipment types val:  ", sorted(val["equipment"].unique()))

Negative weights in validation: 145

                count          mean           std          min           25%  \
distance      12000.0   1141.773100    732.333797     70.00000    556.100000   
weight        11835.0  30506.636248  10572.590450 -47500.00000  25493.000000   
market_index  11751.0      0.926913      0.074109      0.72361      0.863880   
quote_signal  12000.0      2.051334      0.219804      1.23476      1.903467   

                      50%           75%          max  
distance        953.60000   1670.500000   3325.70000  
weight        31193.00000  36843.500000  47500.00000  
market_index      0.92290      0.992355      1.09895  
quote_signal      2.05123      2.200030      2.89635  

Cities in validation but not in train: {'Chicago', 'Laredo', 'Charlotte', 'Knoxville', 'Norfolk', 'San Diego', 'Allentown', 'Jackson'}
Equipment types train: ['Dry Van', 'Flatbed', 'Reefer']
Equipment types val:   ['Dry Van', 'Flatbed', 'Reefer']


In [11]:
tc = train[train["rpm_ratio"] <= 2.0].copy()
tc["weight"] = tc["weight"].abs()
print("Rows kept:", len(tc))
print()

print("Correlation with posted_rate:")
print(tc[["posted_rate", "distance", "weight", "market_index", "quote_signal"]].corr()["posted_rate"])
print()

print("Train by month (medians):")
print(tc.groupby("month")[["rpm", "market_index", "quote_signal"]].median())
print()

val["month"] = pd.to_datetime(val["date"]).dt.month
print("Validation by month (medians):")
print(val.groupby("month")[["market_index", "quote_signal"]].median())
print()

print("Median rate per mile by equipment:")
print(tc.groupby("equipment")["rpm"].median())

Rows kept: 47660

Correlation with posted_rate:
posted_rate     1.000000
distance        0.982263
weight          0.045165
market_index    0.036478
quote_signal   -0.041753
Name: posted_rate, dtype: float64

Train by month (medians):
            rpm  market_index  quote_signal
month                                      
1      2.026936      0.933240      2.028470
2      2.059004      0.993160      2.060790
3      2.137686      1.064915      2.139380
4      2.144748      1.207075      2.003340
5      2.188088      1.304470      1.957185
6      2.244601      1.275230      2.247425
7      2.184772      1.212250      1.963325
8      2.119387      0.982370      2.052790
9      2.156681      0.887880      2.160090
10     2.161176      0.958095      1.985955

Validation by month (medians):
       market_index  quote_signal
month                            
11         0.912290       2.05755
12         0.931515       2.04675

Median rate per mile by equipment:
equipment
Dry Van    2.044431
Flat

In [12]:
print("Validation by month (medians):")
print(val.groupby("month")[["market_index", "quote_signal"]].median())
print()

print("Correlation with rate per mile, within each training month:")
for m, g in tc.groupby("month"):
    print(m, "quote_signal:", round(g["quote_signal"].corr(g["rpm"]), 3),
          "| market_index:", round(g["market_index"].corr(g["rpm"]), 3))

Validation by month (medians):
       market_index  quote_signal
month                            
11         0.912290       2.05755
12         0.931515       2.04675

Correlation with rate per mile, within each training month:
1 quote_signal: 0.873 | market_index: 0.067
2 quote_signal: 0.885 | market_index: 0.081
3 quote_signal: 0.894 | market_index: 0.102
4 quote_signal: -0.86 | market_index: 0.109
5 quote_signal: -0.871 | market_index: 0.061
6 quote_signal: 0.928 | market_index: 0.049
7 quote_signal: -0.901 | market_index: 0.135
8 quote_signal: -0.012 | market_index: 0.06
9 quote_signal: 0.913 | market_index: 0.034
10 quote_signal: -0.866 | market_index: 0.055


In [13]:
tc["week"] = tc["date_dt"].dt.isocalendar().week.astype(int)

weekly = tc.groupby("week").apply(
    lambda g: pd.Series({
        "corr_with_rpm": g["quote_signal"].corr(g["rpm"]),
        "quote_std": g["quote_signal"].std(),
    })
).round(2)

print(weekly.to_string())
print()
print("Validation quote_signal std:", round(val["quote_signal"].std(), 3))

      corr_with_rpm  quote_std
week                          
1              0.92       0.25
2              0.91       0.26
3              0.91       0.25
4              0.81       0.25
5              0.82       0.25
6              0.87       0.24
7              0.89       0.26
8              0.89       0.24
9              0.94       0.26
10             0.87       0.26
11             0.83       0.25
12             0.89       0.29
13             0.92       0.27
14            -0.48       0.27
15            -0.90       0.27
16            -0.89       0.25
17            -0.85       0.25
18            -0.80       0.27
19            -0.86       0.25
20            -0.89       0.27
21            -0.87       0.26
22            -0.62       0.30
23             0.90       0.28
24             0.97       0.29
25             0.93       0.28
26             0.95       0.30
27            -0.62       0.30
28            -0.89       0.27
29            -0.90       0.26
30            -0.92       0.26
31      

In [14]:
dev = train[train["rpm_ratio"] <= 2.0].copy()   # drop the 340 bad-label rows
dev["date_dt"] = pd.to_datetime(dev["date"])
dev["month"] = dev["date_dt"].dt.month

fit = dev[dev["month"] <= 8]
hold = dev[dev["month"] >= 9]
print("Fit rows:", len(fit), "| Holdout rows:", len(hold))

# Baseline: median rate per mile by equipment, times distance
med = fit.groupby("equipment")["rpm"].median()
pred = hold["distance"] * hold["equipment"].map(med)

err = (pred - hold["posted_rate"]).abs()
print("Baseline MAE ($):", round(err.mean(), 1))
print("Baseline MAPE (%):", round((err / hold["posted_rate"]).mean() * 100, 1))

Fit rows: 38210 | Holdout rows: 9450
Baseline MAE ($): 184.8
Baseline MAPE (%): 10.0


In [15]:
import lightgbm as lgb

dev["weight"] = dev["weight"].abs()
dev["weight"] = dev["weight"].fillna(dev.groupby("equipment")["weight"].transform("median"))
dev["market_index"] = dev["market_index"].fillna(dev.groupby("date")["market_index"].transform("median"))
dev["eq"] = dev["equipment"].astype("category").cat.codes
dev["week"] = dev["date_dt"].dt.isocalendar().week.astype(int)

def run(fit, hd, cols):
    m = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=31,
                          min_child_samples=40, subsample=0.8, subsample_freq=1, verbose=-1)
    m.fit(fit[cols], fit["rpm"])
    pred = m.predict(hd[cols]) * hd["distance"]
    err = (pred - hd["posted_rate"]).abs()
    return round(err.mean(), 1), round((err / hd["posted_rate"]).mean() * 100, 2)

basic = ["distance", "eq", "weight"]
coords = ["pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"]
sets = {
    "basic": basic,
    "basic + coords": basic + coords,
    "basic + market_index": basic + ["market_index"],
    "basic + quote_signal": basic + ["quote_signal"],
}

fit = dev[dev["month"] <= 8]
hold = dev[dev["month"] >= 9]
print("Test 1: train Jan-Aug, test Sep-Oct  (MAE, MAPE%)")
for name, cols in sets.items():
    print(f"  {name:22s}", run(fit, hold, cols))

early = dev[dev["week"] < 32]
aug = dev[(dev["week"] >= 32) & (dev["week"] <= 35)]
print()
print("Test 2: train weeks 1-31, test the noisy August weeks")
for name, cols in sets.items():
    print(f"  {name:22s}", run(early, aug, cols))

Test 1: train Jan-Aug, test Sep-Oct  (MAE, MAPE%)
  basic                  (np.float64(73.1), np.float64(4.85))
  basic + coords         (np.float64(60.5), np.float64(4.12))
  basic + market_index   (np.float64(128.2), np.float64(6.9))
  basic + quote_signal   (np.float64(52.0), np.float64(3.71))

Test 2: train weeks 1-31, test the noisy August weeks
  basic                  (np.float64(67.9), np.float64(4.73))
  basic + coords         (np.float64(53.2), np.float64(3.85))
  basic + market_index   (np.float64(86.9), np.float64(5.35))
  basic + quote_signal   (np.float64(121.4), np.float64(7.22))


In [16]:
dev["dow"] = dev["date_dt"].dt.dayofweek
dev["mi_day"] = dev.groupby("date")["market_index"].transform("median")

sets2 = {
    "basic + coords":                  basic + coords,
    "basic + coords + day_of_week":    basic + coords + ["dow"],
    "basic + coords + dow + market":   basic + coords + ["dow", "mi_day"],
}

fit = dev[dev["month"] <= 8]
hold = dev[dev["month"] >= 9]
early = dev[dev["week"] < 32]
aug = dev[(dev["week"] >= 32) & (dev["week"] <= 35)]

print("Test 1 (train Jan-Aug, test Sep-Oct)    |  Test 2 (train wk1-31, test Aug)")
for name, cols in sets2.items():
    a = run(fit, hold, cols)
    b = run(early, aug, cols)
    print(f"{name:32s} MAE {a[0]:6.1f}  MAPE {a[1]:.2f}%  |  MAE {b[0]:6.1f}  MAPE {b[1]:.2f}%")

C:\Users\hp\AppData\Local\Temp\ipykernel_9704\2331598616.py:1: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  dev["dow"] = dev["date_dt"].dt.dayofweek


Test 1 (train Jan-Aug, test Sep-Oct)    |  Test 2 (train wk1-31, test Aug)
basic + coords                   MAE   60.5  MAPE 4.12%  |  MAE   53.2  MAPE 3.85%
basic + coords + day_of_week     MAE   58.4  MAPE 4.05%  |  MAE   50.7  MAPE 3.74%
basic + coords + dow + market    MAE  137.9  MAPE 7.26%  |  MAE   88.7  MAPE 5.31%


In [17]:
import sys
sys.path.insert(0, "../src")
import features as F

raw = pd.read_csv("../data/train_test.csv")
clean = F.drop_bad_labels(raw)
print("Rows kept:", len(clean))

med = F.weight_medians(clean)
ready = F.add_features(clean, med)
print(ready[F.FEATURES].isna().sum())

Rows kept: 47660
distance        0
eq              0
weight          0
pickup_lat      0
pickup_lon      0
delivery_lat    0
delivery_lon    0
dow             0
dtype: int64
